# VERA — Full Ablation Study on **Real** Language-Table Data

This notebook downloads the **official Language-Table dataset** from Google Research,
converts it to our pkl format, then runs 6 ablation conditions × 3 seeds.

### Data download methods (tried in order)
| Method | What it does |
|---|---|
| **Method A** — TFDS | `tfds.load('language_table')` — works if GCS is public |
| **Method B** — gsutil | `gsutil cp gs://gresearch/robotics/language_table/...` |
| **Method C** — Drive cache | Restores from a previous download saved to Drive |
| **Method D** — Synthetic fallback | Rendered synthetic episodes (for testing only) |

### Expected runtime
- Data download + conversion: **30–90 min** (one-time, then cached to Drive)
- Training (A100): **6 conditions × 3 seeds × ~25 min ≈ 7–8 hours**
- Training (T4): **6 conditions × 3 seeds × ~50 min ≈ 15 hours**

### Recommended runtime
Runtime → Change runtime type → **A100** (if available) or **T4**

In [ ]:
# ── Cell 1: GPU + RAM check ──────────────────────────────────────────────────
import torch, psutil, os

cuda_ok = torch.cuda.is_available()
print(f'CUDA GPU : {cuda_ok}')
if cuda_ok:
    print(f'  Device : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('  ❌  No CUDA GPU — Runtime → Change runtime type → T4 GPU')
    raise SystemExit('Switch to a GPU runtime and re-run.')

print(f'RAM  : {psutil.virtual_memory().total/1e9:.1f} GB total  |  '
      f'{psutil.virtual_memory().available/1e9:.1f} GB free')
print('GPU check passed ✓')

In [ ]:
# ── Cell 2: Mount Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VERA_LT_Real'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/lt_real_data', exist_ok=True)
print('Drive mounted ✓  →', DRIVE_ROOT)

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# Core ML deps
!pip install -q git+https://github.com/openai/CLIP.git ftfy regex pyyaml

# Language-Table data access
!pip install -q tensorflow tensorflow_datasets rlds

# language_table package (environment + dataset utils from Google Research)
!pip install -q language_table

import clip, torch, yaml
import tensorflow as tf
print('torch:', torch.__version__, '  CLIP ready ✓')
print('tensorflow:', tf.__version__, ' ✓')

# Suppress TF warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

In [ ]:
# ── Cell 4: Clone / pull repo ────────────────────────────────────────────────
import sys

REPO_URL = 'https://github.com/sara-kaz/RLConditionedVLA.git'
REPO_DIR = '/content/RLConditionedVLA'

if os.path.exists(REPO_DIR):
    result = !cd {REPO_DIR} && git pull
    print('\n'.join(result))
else:
    !git clone -q {REPO_URL} {REPO_DIR}
    print('Cloned ✓')

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# Force-clear stale module cache after git pull
for mod in list(sys.modules.keys()):
    if any(k in mod for k in ('vera_model', 'trajectory_dataset', 'sft_trainer')):
        del sys.modules[mod]

# ── Verify E_exp fix ──────────────────────────────────────────────────────────
import numpy as np
from models.vera_model import verbalize_consequence

lt_rewards   = [0.061, 0.061, 0.080, 0.163, 0.080, 0.0002, 0.0002]
delta_r      = np.zeros(len(lt_rewards))
delta_r[1:]  = np.diff(lt_rewards)
state_deltas = np.clip(-delta_r * 3.0, -0.5, 0.5)

seen = set()
print('E_exp string diversity check (should see ≥ 5 distinct strings):')
for r, d in zip(lt_rewards, state_deltas):
    s = verbalize_consequence(float(r), float(d))
    seen.add(s)
    print(f'  r={r:.4f}  δ={d:+.3f}  →  {s}')

n_distinct = len(seen)
if n_distinct >= 5:
    print(f'\n  ✓  {n_distinct} distinct strings — fix confirmed')
else:
    print(f'\n  ❌  Only {n_distinct} strings — git pull may not have updated the file')
    !cd {REPO_DIR} && git log --oneline -5

print('\nRepo:', os.getcwd())

In [ ]:
# ── Cell 5: Download & Convert Real Language-Table Data ──────────────────────
#
# Tries three methods in order:
#   A) TFDS   — tfds.load('language_table') direct streaming
#   B) gsutil — copy TFRecord files from gs://gresearch/robotics/language_table/
#   C) Drive  — restore previously downloaded/converted pkl episodes from Drive
#
# Once converted, episodes are saved as episode_XXXXX/steps.pkl and
# archived to Drive so this cell can be skipped on re-runs.
#
# ── Configuration ─────────────────────────────────────────────────────────────
MAX_EPISODES    = 3000    # how many real episodes to use (3k is enough for ablation)
                          # full LT train set has ~150k episodes
VAL_FRACTION    = 0.10    # fraction held out for validation
LOCAL_LT_TRAIN  = '/content/lt_real/train'
DRIVE_LT_TRAIN  = f'{DRIVE_ROOT}/lt_real_data/train'
DRIVE_TAR_PATH  = f'{DRIVE_ROOT}/lt_real_data/lt_real_{MAX_EPISODES}eps.tar'
# ─────────────────────────────────────────────────────────────────────────────

import pickle, tarfile, shutil, math
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

os.makedirs(LOCAL_LT_TRAIN, exist_ok=True)

def count_ep(d):
    return len(list(Path(d).glob('episode_*'))) if Path(d).exists() else 0

def action_to_bin(dx, dy, num_bins=8):
    """Convert continuous 2D delta → discrete bin index (arctan2 sectors)."""
    angle = math.atan2(dy, dx)             # [-π, π]
    angle = angle % (2 * math.pi)          # [0, 2π]
    bin_w = (2 * math.pi) / num_bins
    return int(angle / bin_w) % num_bins

def save_episode_pkl(steps_list, ep_idx, out_dir):
    """Save one episode as episode_XXXXX/steps.pkl in our loader format."""
    ep_dir = Path(out_dir) / f'episode_{ep_idx:05d}'
    ep_dir.mkdir(parents=True, exist_ok=True)
    with open(ep_dir / 'steps.pkl', 'wb') as f:
        pickle.dump(steps_list, f)

def convert_rlds_episode(ep_steps, ep_idx, out_dir):
    """
    Convert one RLDS episode (list of step dicts from TFDS) to our pkl format.
    Each step becomes a dict with keys: obs (rgb), action, reward, instruction.
    """
    steps_list = []
    for step in ep_steps:
        obs = step.get('observation', step)  # handle flat or nested

        # ── Image ────────────────────────────────────────────────────────────
        if 'rgb' in obs:
            rgb = obs['rgb']
        elif 'image' in obs:
            rgb = obs['image']
        else:
            rgb = list(obs.values())[0]   # best guess
        if hasattr(rgb, 'numpy'): rgb = rgb.numpy()
        rgb = np.array(rgb, dtype=np.uint8)

        # ── Instruction ───────────────────────────────────────────────────────
        instr_raw = obs.get('instruction', obs.get('language_instruction', b'complete the task'))
        if hasattr(instr_raw, 'numpy'): instr_raw = instr_raw.numpy()
        instruction = instr_raw.decode('utf-8') if isinstance(instr_raw, bytes) else str(instr_raw)

        # ── Action (2D delta → keep both continuous and discrete) ─────────────
        action_raw = step.get('action', np.zeros(2, dtype=np.float32))
        if hasattr(action_raw, 'numpy'): action_raw = action_raw.numpy()
        action_vec = np.array(action_raw, dtype=np.float32).flatten()[:2]  # [dx, dy]
        action_bin = action_to_bin(float(action_vec[0]), float(action_vec[1]))

        # ── Reward ───────────────────────────────────────────────────────────
        reward_raw = step.get('reward', 0.0)
        if hasattr(reward_raw, 'numpy'): reward_raw = reward_raw.numpy()
        reward = float(reward_raw)

        steps_list.append({
            'obs':          {'rgb': rgb},
            'action':       action_vec,       # continuous [dx, dy] — loader uses this
            'action_bin':   action_bin,       # precomputed discrete index (optional)
            'reward':       reward,
            'instruction':  instruction,
        })

    if len(steps_list) >= 4:
        save_episode_pkl(steps_list, ep_idx, out_dir)
        return True
    return False

# ─────────────────────────────────────────────────────────────────────────────
# Check existing converted data
# ─────────────────────────────────────────────────────────────────────────────
n_existing = count_ep(LOCAL_LT_TRAIN)
if n_existing >= MAX_EPISODES:
    print(f'✓ Already have {n_existing} episodes in {LOCAL_LT_TRAIN} — skipping download')
    LT_DATA_PATH = LOCAL_LT_TRAIN

# ─────────────────────────────────────────────────────────────────────────────
# Method C: restore from Drive tar (fastest if previously downloaded)
# ─────────────────────────────────────────────────────────────────────────────
elif os.path.exists(DRIVE_TAR_PATH):
    print(f'Restoring {MAX_EPISODES} episodes from Drive tar...')
    with tarfile.open(DRIVE_TAR_PATH, 'r') as t:
        t.extractall('/content/')
    n_restored = count_ep(LOCAL_LT_TRAIN)
    print(f'  Restored {n_restored} episodes ✓')
    LT_DATA_PATH = LOCAL_LT_TRAIN

else:
    # ─────────────────────────────────────────────────────────────────────────
    # Method A: TFDS direct load
    # ─────────────────────────────────────────────────────────────────────────
    import tensorflow_datasets as tfds

    TFDS_NAMES = [
        'language_table',
        'language_table_sim',
        'language_table/language_table',
    ]

    tfds_ok = False
    for ds_name in TFDS_NAMES:
        try:
            print(f'Trying TFDS: {ds_name} ...')
            builder = tfds.builder(ds_name)
            builder.download_and_prepare(
                download_dir='/content/lt_tfds_cache/',
                download_config=tfds.download.DownloadConfig(verify_ssl=False)
            )
            ds = builder.as_dataset(split='train')
            tfds_ok = True
            tfds_name_used = ds_name
            print(f'  TFDS load succeeded: {ds_name} ✓')
            break
        except Exception as e:
            print(f'  failed: {e}')

    if tfds_ok:
        print(f'\nConverting {MAX_EPISODES} episodes from TFDS...')
        ep_idx = 0
        pbar   = tqdm(total=MAX_EPISODES, desc='converting')

        # RLDS format: each element is an episode dict with 'steps' key
        for ep in ds.as_numpy_iterator():
            if ep_idx >= MAX_EPISODES: break
            steps_raw = list(ep['steps']) if 'steps' in ep else [ep]
            if convert_rlds_episode(steps_raw, ep_idx, LOCAL_LT_TRAIN):
                ep_idx += 1
                pbar.update(1)
        pbar.close()
        print(f'Converted {ep_idx} episodes ✓')
        LT_DATA_PATH = LOCAL_LT_TRAIN

    else:
        # ─────────────────────────────────────────────────────────────────────
        # Method B: gsutil from GCS
        # ─────────────────────────────────────────────────────────────────────
        print('\nTFDS failed. Trying gsutil from Google Cloud Storage...')
        print('You may need to authenticate with Google Cloud first.')

        try:
            from google.colab import auth
            auth.authenticate_user()
            print('  Google Cloud authenticated ✓')
        except Exception as e:
            print(f'  Auth skipped: {e}')

        GCS_PATH     = 'gs://gresearch/robotics/language_table'
        RAW_DIR      = '/content/lt_raw_tfrecord/'
        os.makedirs(RAW_DIR, exist_ok=True)

        print(f'  Downloading TFRecord files from {GCS_PATH} ...')
        print(f'  (downloading ~5 shards ≈ 3–8 GB, takes 10–20 min)')
        !gsutil -q -m cp \
            "gs://gresearch/robotics/language_table/0.0.1/language_table-train.tfrecord-00000-of-*" \
            {RAW_DIR}

        tfrecord_files = sorted(list(Path(RAW_DIR).glob('*.tfrecord*')))
        print(f'  Downloaded {len(tfrecord_files)} TFRecord files')

        if len(tfrecord_files) == 0:
            print('\n  ❌  gsutil download failed. Options:')
            print('     1. Run:  !gcloud auth application-default login --no-launch-browser')
            print('        then re-run this cell.')
            print('     2. Or manually upload your Language-Table pkl episodes to')
            print('        Google Drive → VERA_LT_Real/lt_real_data/train/')
            print('        and set LT_DATA_PATH manually below.')
            print('\n  Falling back to synthetic data for pipeline testing...')

            # ─────────────────────────────────────────────────────────────────
            # Method D: Synthetic fallback (testing only)
            # ─────────────────────────────────────────────────────────────────
            from PIL import ImageDraw
            IMG_SZ = 64
            TRAIN_N, EP_LEN = 900, 20
            DIR_NAMES = [
                'to the right', 'up and to the right', 'upward', 'up and to the left',
                'to the left', 'down and to the left', 'downward', 'down and to the right',
            ]
            SYNTH_DIR = Path('/content/lt_synth_fallback/train')
            SYNTH_DIR.mkdir(parents=True, exist_ok=True)

            def render_frame(bx, by, gx, gy):
                img = Image.new('RGB', (IMG_SZ, IMG_SZ), (235, 235, 235))
                d   = ImageDraw.Draw(img)
                d.ellipse([int(gx)-6, int(gy)-6, int(gx)+6, int(gy)+6], fill=(30, 180, 30))
                d.ellipse([int(bx)-8, int(by)-8, int(bx)+8, int(by)+8], fill=(210, 40, 40))
                return np.array(img, dtype=np.uint8)

            for i in tqdm(range(TRAIN_N), desc='synth fallback'):
                b = 18; T = EP_LEN + np.random.randint(-3, 6)
                bin_id = i % 8
                angle  = bin_id * (2 * np.pi / 8)
                spread = float(np.random.uniform(18, 26))
                bx = float(np.random.randint(b, IMG_SZ - b))
                by = float(np.random.randint(b, IMG_SZ - b))
                gx = float(np.clip(bx + np.cos(angle) * spread, b, IMG_SZ - b))
                gy = float(np.clip(by + np.sin(angle) * spread, b, IMG_SZ - b))
                steps_list = []
                for _ in range(T):
                    dx = gx - bx; dy = gy - by
                    dist = float(np.sqrt(dx*dx + dy*dy)) + 1e-6
                    act  = np.array([dx/dist * 0.8 + np.random.normal(0, 0.04),
                                     dy/dist * 0.8 + np.random.normal(0, 0.04)],
                                    dtype=np.float32).clip(-1, 1)
                    rew  = float(min(1.0, max(0.0, 1.0 - dist / (IMG_SZ * 0.5))))
                    steps_list.append({
                        'obs':         {'rgb': render_frame(bx, by, gx, gy)},
                        'action':      act,
                        'reward':      rew,
                        'instruction': f'push the block {DIR_NAMES[bin_id]}',
                    })
                    step_px = 4.0
                    bx = float(np.clip(bx + dx/dist*step_px + np.random.normal(0, 0.5),
                                       b//2, IMG_SZ - b//2))
                    by = float(np.clip(by + dy/dist*step_px + np.random.normal(0, 0.5),
                                       b//2, IMG_SZ - b//2))
                save_episode_pkl(steps_list, i, str(SYNTH_DIR))

            LT_DATA_PATH = str(SYNTH_DIR)
            print(f'  Synthetic fallback ready: {TRAIN_N} episodes')
            print('  ⚠  These are NOT real LT episodes — results will match previous run')

        else:
            # Parse TFRecords into episodes
            import tensorflow as tf

            print('\nParsing TFRecord files into episodes...')
            raw_ds = tf.data.TFRecordDataset(str(f) for f in tfrecord_files)

            ep_idx     = 0
            cur_steps  = []
            cur_ep_id  = None
            pbar       = tqdm(total=MAX_EPISODES, desc='converting')

            for raw in raw_ds:
                if ep_idx >= MAX_EPISODES: break
                try:
                    ex = tf.train.Example()
                    ex.ParseFromString(raw.numpy())
                    f  = ex.features.feature

                    def get_bytes(key):
                        return f[key].bytes_list.value[0] if key in f else b''
                    def get_float(key):
                        v = f[key].float_list.value
                        return np.array(list(v), dtype=np.float32) if v else np.zeros(1, dtype=np.float32)
                    def get_int(key):
                        v = f[key].int64_list.value
                        return int(v[0]) if v else 0

                    ep_id   = get_bytes('episode_id')
                    is_last = bool(get_int('is_last'))

                    # Try to decode image
                    img_bytes = get_bytes('image') or get_bytes('rgb')
                    if img_bytes:
                        img_tensor = tf.image.decode_jpeg(img_bytes, channels=3)
                        rgb = img_tensor.numpy().astype(np.uint8)
                    else:
                        rgb = np.zeros((64, 64, 3), dtype=np.uint8)

                    instr_raw   = get_bytes('instruction') or get_bytes('language_instruction')
                    instruction = instr_raw.decode('utf-8') if instr_raw else 'complete the task'

                    action_vals = get_float('action') if 'action' in f else np.zeros(2)
                    action_vec  = action_vals[:2].astype(np.float32)
                    reward_arr  = get_float('reward')
                    reward      = float(reward_arr[0]) if len(reward_arr) > 0 else 0.0

                    cur_steps.append({
                        'obs':         {'rgb': rgb},
                        'action':      action_vec,
                        'reward':      reward,
                        'instruction': instruction,
                    })

                    if is_last or (ep_id != cur_ep_id and cur_ep_id is not None):
                        if len(cur_steps) >= 4:
                            save_episode_pkl(cur_steps, ep_idx, LOCAL_LT_TRAIN)
                            ep_idx += 1
                            pbar.update(1)
                        cur_steps  = []
                    cur_ep_id = ep_id

                except Exception as e:
                    cur_steps = []
                    continue

            pbar.close()
            print(f'Converted {ep_idx} episodes from TFRecord ✓')
            LT_DATA_PATH = LOCAL_LT_TRAIN

    # ── Save converted data to Drive (avoid re-downloading next session) ───────
    n_final = count_ep(LOCAL_LT_TRAIN)
    if n_final > 0:
        print(f'\nArchiving {n_final} episodes to Drive...')
        local_tar = '/content/lt_real_tmp.tar'
        with tarfile.open(local_tar, 'w') as t:
            for p in tqdm(Path(LOCAL_LT_TRAIN).rglob('*.pkl'), desc='pack'):
                t.add(str(p), arcname=str(p.relative_to('/content/')))
        shutil.copy2(local_tar, DRIVE_TAR_PATH)
        os.remove(local_tar)
        sz_mb = os.path.getsize(DRIVE_TAR_PATH) / 1e6
        print(f'  Saved to Drive: {DRIVE_TAR_PATH}  ({sz_mb:.0f} MB)')
        print('  Next session: Cell 5 will restore in ~2 min instead of re-downloading')

# ─────────────────────────────────────────────────────────────────────────────
n_final = count_ep(LOCAL_LT_TRAIN)
print(f'\n✓ Data ready: {n_final} episodes at {LT_DATA_PATH}')

In [ ]:
# ── Cell 6: Smoke test loader ─────────────────────────────────────────────────
from data.trajectory_dataset import load_language_table, TrajectoryDataset
import psutil

eps = load_language_table(LT_DATA_PATH)
assert len(eps) > 0, f'No episodes loaded from {LT_DATA_PATH}'

e0 = eps[0]
print(f'Episodes loaded : {len(eps)}')
print(f'Frames shape    : {e0["frames"].shape}')
print(f'Actions shape   : {e0["actions"].shape}  (8-bin discrete)')
print(f'ActionVec shape : {e0["action_vectors"].shape}  (expect T×2)')
print(f'state_deltas    : {"YES " + str(e0["state_deltas"].shape) if "state_deltas" in e0 else "MISSING"}')
print(f'Instruction     : {e0["instruction"]}')
print(f'Rewards range   : [{e0["rewards"].min():.4f}, {e0["rewards"].max():.4f}]')
print(f'RAM used        : {psutil.virtual_memory().used/1e9:.1f} GB')

if 'state_deltas' not in e0:
    raise RuntimeError('state_deltas missing — git pull did not update trajectory_dataset.py')

# Quick batch test
ds = TrajectoryDataset(eps[:10], history_len=4, num_vis_frames=3,
                       num_actions=8, action_dim=2, img_size=64, device='cpu')
batch0 = ds[0]
print(f'\nBatch keys      : {list(batch0.keys())}')
assert 'state_delta' in batch0, 'state_delta missing from batch'
print(f'state_delta val : {batch0["state_delta"].item():.4f}')

# Show action distribution (checks data is not all one bin)
from collections import Counter
bin_counts = Counter()
for ep in eps[:200]:
    for a in ep['actions']:
        bin_counts[int(a)] += 1
total = sum(bin_counts.values())
print('\nAction bin distribution (expect spread across all 8 bins):')
for b in range(8):
    bar = '█' * int(bin_counts[b] / total * 40)
    print(f'  bin {b}: {bar} {bin_counts[b]/total*100:.1f}%')

print('\nLoader smoke test ✓')

In [ ]:
# ── Cell 7: Build base config ─────────────────────────────────────────────────
import yaml, copy

with open(f'{REPO_DIR}/configs/config.yaml') as f:
    base_cfg = yaml.safe_load(f)

base_cfg['data']['episodes_path'] = LT_DATA_PATH
base_cfg['data']['dataset_type']  = 'language_table'
base_cfg['data']['img_size']      = 224

base_cfg['training'].update({
    'output_dir'  : f'{DRIVE_ROOT}/checkpoints',
    'device'      : 'auto',
    'batch_size'  : 32,
    'num_workers' : 2,
    'save_every'  : 25,
})

print('Config ready ✓')
print(f'  episodes_path        : {base_cfg["data"]["episodes_path"]}')
print(f'  dataset_type         : {base_cfg["data"]["dataset_type"]}')
print(f'  action_dim           : {base_cfg["model"]["action_dim"]}  (expect 2 for LT)')
print(f'  unfreeze_clip_vision : {base_cfg["model"]["unfreeze_clip_vision"]}')
print(f'  alignment_loss_coef  : {base_cfg["vera"]["alignment_loss_coef"]}  (expect 0.15)')
print(f'  regression_loss_coef : {base_cfg["vera"]["regression_loss_coef"]}')
print(f'  lr                   : {base_cfg["training"]["lr"]}')
print(f'  epochs               : {base_cfg["training"]["epochs"]}')
print(f'  early_stopping       : {base_cfg["training"]["early_stopping_patience"]} epochs')
print(f'  clip_vision_lr       : {base_cfg["training"]["clip_vision_lr"]}')
print(f'  action_vocab entries : {len(base_cfg["vera"]["action_vocab"])}')

if base_cfg['vera']['alignment_loss_coef'] == 0.0:
    print('\n  ❌  alignment_loss_coef is 0.0 — forcing to 0.15')
    base_cfg['vera']['alignment_loss_coef'] = 0.15

In [ ]:
# ── Cell 8: Run 6 conditions × 3 seeds ───────────────────────────────────────
import copy, gc, json as _json
import numpy as np, torch, psutil
from training.sft_trainer_vera import sft_train

SEEDS = [42, 123, 456]

CONDITIONS = [
    {
        'name'    : 'full_vera',
        'display' : 'Full VERA (all 5 streams)',
        'overrides': {},
    },
    {
        'name'    : 'bc_baseline',
        'display' : 'BC baseline (no lang, no hist TF)',
        'overrides': {
            ('vera', 'use_lang_feedback')    : False,
            ('vera', 'use_consequence_token'): False,
            ('vera', 'use_temporal_history') : False,
        },
    },
    {
        'name'    : 'no_lang',
        'display' : 'No language feedback (hist TF + gate kept)',
        'overrides': {
            ('vera', 'use_lang_feedback')    : False,
            ('vera', 'use_consequence_token'): False,
        },
    },
    {
        'name'    : 'no_exp',
        'display' : 'No E_exp / Stream 3b (action narration only)',
        'overrides': {
            ('vera', 'use_consequence_token'): False,
        },
    },
    {
        'name'    : 'no_act',
        'display' : 'No E_act / Stream 3a (experience token only)',
        'overrides': {
            ('vera', 'use_lang_feedback'): False,
        },
    },
    {
        'name'    : 'no_hist_tf',
        'display' : 'No history sub-transformer (flat positional hist)',
        'overrides': {
            ('vera', 'use_temporal_history'): False,
        },
    },
]

all_results = {}

for ci, cond in enumerate(CONDITIONS):
    cond_results = {}
    print(f'\n{"═"*65}')
    print(f'  [{ci+1}/{len(CONDITIONS)}]  {cond["display"]}')
    print(f'{"═"*65}')

    for seed in SEEDS:
        run_dir  = f'{DRIVE_ROOT}/checkpoints/lt_{cond["name"]}/seed{seed}'
        log_path = f'{run_dir}/sft_vera_log.json'

        if os.path.exists(log_path):
            log = _json.load(open(log_path))
            if log:
                best = max(r['val_acc'] for r in log)
                print(f'  [SKIP] seed={seed}  best val_acc={best:.4f}')
                cond_results[f'seed{seed}'] = best
                continue

        ram_gb = psutil.virtual_memory().used / 1e9
        print(f'\n  seed={seed}  RAM={ram_gb:.1f} GB')
        os.makedirs(run_dir, exist_ok=True)

        cfg = copy.deepcopy(base_cfg)
        for (sec, key), val in cond['overrides'].items():
            cfg[sec][key] = val
        cfg['training']['output_dir'] = run_dir

        torch.manual_seed(seed)
        np.random.seed(seed)
        sft_train(cfg)

        torch.cuda.empty_cache()
        gc.collect()

        best = 0.0
        if os.path.exists(log_path):
            log = _json.load(open(log_path))
            best = max(r['val_acc'] for r in log) if log else 0.0
        cond_results[f'seed{seed}'] = best
        print(f'  ✓ seed={seed}  best val_acc={best:.4f}')

    all_results[cond['name']] = cond_results

summary = {'results': all_results, 'data_source': LT_DATA_PATH}
_json.dump(summary, open(f'{DRIVE_ROOT}/checkpoints/lt_real_summary.json', 'w'), indent=2)
print(f'\n✓ All conditions done. Summary saved to Drive.')

In [ ]:
# ── Cell 9: Paper-ready results table ────────────────────────────────────────
import json, numpy as np

data = json.load(open(f'{DRIVE_ROOT}/checkpoints/lt_real_summary.json'))
res  = data['results']
src  = data.get('data_source', 'unknown')
print(f'Data source: {src}\n')

ORDER = [
    ('bc_baseline', 'BC/SFT baseline              (no lang, no hist TF)'),
    ('no_lang',     'No lang feedback†             (hist TF + gate kept)'),
    ('no_hist_tf',  'No history sub-transformer    (flat positional hist)'),
    ('no_act',      'No E_act / Stream 3a          (experience token only)'),
    ('no_exp',      'No E_exp / Stream 3b          (action narration only)'),
    ('full_vera',   'Full VERA ★                   (all 5 streams)'),
]

print('─'*82)
print('Language-Table ablation — validation action-classification accuracy (%)')
print(f'{"Variant":<54} {"S42":>6} {"S123":>6} {"S456":>6}  {"Mean±Std":>12}')
print('─'*82)

vera_mu, bc_mu = None, None
for key, display in ORDER:
    if key not in res:
        print(f'  {display:<54}  (not run yet)')
        continue
    v  = [res[key].get(f'seed{s}', float('nan')) for s in [42, 123, 456]]
    mu = float(np.nanmean(v))
    sd = float(np.nanstd(v))
    if key == 'full_vera' : vera_mu = mu
    if key == 'bc_baseline': bc_mu  = mu
    vals   = '  '.join(f'{x*100:5.1f}' if not np.isnan(x) else '  ---' for x in v)
    marker = '  ← best' if key == 'full_vera' else ''
    print(f'  {display:<54}  {vals}   {mu*100:.1f}±{sd*100:.1f}{marker}')

print('─'*82)

if vera_mu is not None and bc_mu is not None:
    delta = (vera_mu - bc_mu) * 100
    sign  = '+' if delta >= 0 else ''
    print(f'  VERA vs BC delta: {sign}{delta:.2f} pp')

print()
print('† Removes E_act (Stream 3a) and E_exp (Stream 3b); hist TF and reward gate remain.')
print()
print('Copy these numbers into vera_corl.tex → Table 1 (ablations).')

In [ ]:
# ── Cell 10: Download results zip ─────────────────────────────────────────────
from google.colab import files
import shutil

shutil.make_archive('/content/vera_lt_real_results', 'zip', f'{DRIVE_ROOT}/checkpoints')
files.download('/content/vera_lt_real_results.zip')
print('Download started ✓')